In [1]:
import tensorflow as tf
if tf.__version__ != "2.14.0":
    print(f"Current TensorFlow version: {tf.__version__}, switching to 2.14.0")

    # Uninstall current TensorFlow version
    !pip uninstall -y tensorflow

    # Install TensorFlow 2.10
    !pip install numpy==1.26 --force-reinstall
    !pip install tensorflow==2.14.0


    # After installation, restart runtime
    print("TensorFlow 2.14.0 installed.")
    print("Please click on the Runtime > Restart session and run all.")
else:
    print("TensorFlow 2.14.0 is already installed.")

TensorFlow 2.14.0 is already installed.


In [2]:
# * tensorflow para construir y entrenar la red neuronal.
# * numpy para manejar los datos numéricos.
# * Sequential y Dense de keras para definir y estructurar la red neuronal.
import tensorflow as tf
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [3]:
# Creación del modelo
# Crea una capa con 1 neurona, que recibe 1 entrada.
l0 = Dense(units=1, input_shape=[1])
# Define un modelo secuencial con una sola capa (l0).
model = Sequential([l0])

# Configuración del modelo
# * Se usa el optimizador 'sgd' (descenso de gradiente estocástico) para mejorar los pesos del modelo.
# * Se define la función de pérdida como 'mean_squared_error' (error cuadrático medio) para medir la precisión del modelo.
model.compile(optimizer='sgd', loss='mean_squared_error')

# Datos de entrenamiento
# * xs: valores de entrada.
# * ys: valores de salida esperados
xs = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
ys = np.array([-3.0, -1.0, 1.0, 3.0, 5.0, 7.0], dtype=float)

# Entrenamiento del modelo
model.fit(xs, ys, epochs=500)

# Predicción y pesos aprendidos
# Predice el valor para x = 10.0
print(model.predict(np.array([10.0])))
# Muestra los pesos aprendidos por la red neuronal.
print("Here is what I learned: {}".format(l0.get_weights()))

Epoch 1/500
1/1 [==============================] - 0s 232ms/step - loss: 57.8501
Epoch 2/500
1/1 [==============================] - 0s 7ms/step - loss: 45.9370
Epoch 3/500
1/1 [==============================] - 0s 10ms/step - loss: 36.5557
Epoch 4/500
1/1 [==============================] - 0s 6ms/step - loss: 29.1664
Epoch 5/500
1/1 [==============================] - 0s 6ms/step - loss: 23.3445
Epoch 6/500
1/1 [==============================] - 0s 5ms/step - loss: 18.7559
Epoch 7/500
1/1 [==============================] - 0s 5ms/step - loss: 15.1379
Epoch 8/500
1/1 [==============================] - 0s 5ms/step - loss: 12.2835
Epoch 9/500
1/1 [==============================] - 0s 7ms/step - loss: 10.0302
Epoch 10/500
1/1 [==============================] - 0s 8ms/step - loss: 8.2499
Epoch 11/500
1/1 [==============================] - 0s 8ms/step - loss: 6.8418
Epoch 12/500
1/1 [==============================] - 0s 5ms/step - loss: 5.7268
Epoch 13/500
1/1 [==============================]

In [4]:
# Guarda el modelo en el directorio 'saved_model/1'
export_dir = 'saved_model/1'
tf.saved_model.save(model, export_dir)

In [6]:
# Conversión a TensorFlow Lite
# Convierte el modelo a TensorFlow Lite para ejecutarlo en dispositivos móviles o embebidos.

# Esta línea crea un convertidor (converter) para transformar un modelo guardado de TensorFlow (export_dir) en un formato más ligero
# llamado TensorFlow Lite.
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
# Aquí, el método .convert() ejecuta la conversión y genera el modelo en formato TensorFlow Lite. Es decir:
# * Convierte el modelo a un formato optimizado para dispositivos móviles o embebidos.
# * Reduce el tamaño del modelo, eliminando datos innecesarios.
# * Optimiza la ejecución, permitiendo cálculos más eficientes.
tflite_model = converter.convert()

In [7]:
# Guarda el modelo convertido en un archivo model.tflite
import pathlib                                    # Se importa pathlib, una biblioteca que facilita la manipulación de rutas y archivos.
tflite_model_file = pathlib.Path('model.tflite')  # Se crea un objeto de tipo Path con la ruta 'model.tflite', que representa el
                                                  # archivo donde se almacenará el modelo convertido.
tflite_model_file.write_bytes(tflite_model)       # Se escribe el contenido del modelo (tflite_model) en el archivo 'model.tflite',
                                                  # guardándolo en formato binario.
# El número 1080 que ves en la salida probablemente representa la cantidad de bytes que se escribieron en el archivo 'model.tflite'.

1080

In [11]:
# Load TFLite model and allocate tensors.
# Carga del modelo convertido
# * tf.lite.Interpreter: Es la clase que permite ejecutar modelos TensorFlow Lite en dispositivos embebidos o móviles.
# * model_content=tflite_model: Indica que el modelo que queremos usar es tflite_model, el cual ya fue convertido a formato TensorFlow Lite.
# * Esto crea un objeto interpreter, que nos permite interactuar con el modelo.
interpreter = tf.lite.Interpreter(model_content=tflite_model)
# Asignación de tensores
# * Esto asigna memoria a los tensores de entrada y salida dentro del intérprete.
# * Permite que el modelo esté listo para recibir datos y generar predicciones.
interpreter.allocate_tensors()

# Get input and output tensors.
# Obtención de detalles de entrada y salida
# Muestra información sobre los tensores de entrada y salida del modelo.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(input_details)
print(output_details)

[{'name': 'serving_default_dense_input:0', 'index': 0, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall:0', 'index': 3, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [10]:
# Analicemos lo que representa la salida de los dos prints:
# 1️⃣ print(input_details)
# Este diccionario describe el tensor de entrada del modelo TensorFlow Lite:
# * name: 'serving_default_dense_input:0' → Nombre del tensor de entrada en el modelo guardado.
# * index: 0 → Índice del tensor dentro del intérprete de TensorFlow Lite.
# * shape: [1, 1] → La entrada tiene una forma (1,1), es decir, acepta un único valor a la vez.
# * shape_signature: [-1, 1] → -1 indica que el modelo puede aceptar un número variable de muestras, cada una con una dimensión 1.
# * dtype: numpy.float32 → El tipo de dato esperado es float32.
# * quantization: (0.0, 0) → Indica que no se aplicó cuantización (optimización para reducir el tamaño del modelo).
# * quantization_parameters y sparsity_parameters → Son estructuras vacías porque el modelo no ha sido cuantizado ni usa técnicas de
#                                                   sparsidad.
# 2️⃣ print(output_details)
# Ahora, esta información corresponde al tensor de salida del modelo:
# * name: 'StatefulPartitionedCall:0' → Nombre del tensor de salida asignado por TensorFlow Lite.
# * index: 3 → Índice del tensor de salida dentro del intérprete de TensorFlow Lite.
# * shape: [1, 1] → La salida tiene la misma forma que la entrada (1,1), indicando que por cada valor de entrada, el modelo devuelve una
#                   única predicción.
# * dtype: numpy.float32 → La salida también es un número en float32.
# * Los demás parámetros son idénticos a los del tensor de entrada, indicando que el modelo no ha sido cuantizado ni optimizado con
#   sparsidad.

# Sparsidad (o sparsity en inglés) en el contexto de redes neuronales se refiere a la presencia de muchos valores cero en los pesos del
# modelo. Es una técnica utilizada para optimizar el rendimiento de redes neuronales al reducir el número de cálculos innecesarios.

In [12]:
# Predicción con TensorFlow Lite
# * Se crea un valor de entrada (10.0).
# * Se asigna el valor al tensor de entrada.
# * Se ejecuta la inferencia con interpreter.invoke().
# * Se obtiene el resultado y se imprime.
to_predict = np.array([[10.0]], dtype=np.float32)
print(to_predict)
interpreter.set_tensor(input_details[0]['index'], to_predict)
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])
print(tflite_results)

[[10.]]
[[18.975618]]
